# Part III: Feature Engineering

## Basic settings

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
pd.options.display.max_rows = 50
pd.options.display.max_columns = None
sns.set_theme(style="whitegrid")

In [3]:
src_path = os.path.abspath(os.path.join("..", "src"))
if src_path not in sys.path:
    sys.path.append(src_path)

In [4]:
DATA_DIR = "../data"

# 28-day forecast horizon, evaluated on the last three months of the sample.
HORIZON = 28
CUTOFF_DATE = pd.Timestamp("2025-10-01")

## Load preprocessed data

In [5]:
df_sales = pd.read_csv(
    os.path.join(DATA_DIR, "sales_data_preprocessed.csv"), parse_dates=["date"]
)
df_weather = pd.read_csv(
    os.path.join(DATA_DIR, "weather_preprocessed.csv"), parse_dates=["date"]
)
df_holiday = pd.read_csv(
    os.path.join(DATA_DIR, "holiday_preprocessed.csv"), parse_dates=["date"]
)
df_promotion = pd.read_csv(
    os.path.join(DATA_DIR, "promotion_preprocessed.csv"),
    parse_dates=["start_date", "end_date"],
)

for name, frame in [("sales", df_sales), ("weather", df_weather),
                    ("holiday", df_holiday), ("promotion", df_promotion)]:
    print(f"{name:10s} {len(frame):>8,} rows x {frame.shape[1]:>2} cols")

sales       394,560 rows x 14 cols
weather       4,384 rows x 10 cols
holiday       4,384 rows x  9 cols
promotion     7,177 rows x 12 cols


## Train / test split

**Split before engineering anything that estimates a parameter from the data.**
Bin edges, encodings and scalers are all fitted objects: computing them on the full
frame lets information from the test window leak into training, which inflates
accuracy and is the first thing a reviewer checks.

The split is by date, never random. Data runs 2023-01-01 to 2025-12-31, so the test
window is the final quarter.

In [6]:
df_features = df_sales.copy()
df_features["is_test"] = df_features["date"] >= CUTOFF_DATE

n_train = int((~df_features["is_test"]).sum())
n_test = int(df_features["is_test"].sum())
print(f"cutoff        : {CUTOFF_DATE.date()}")
print(f"train rows    : {n_train:,}  ({df_features.loc[~df_features['is_test'], 'date'].min().date()} "
      f"to {df_features.loc[~df_features['is_test'], 'date'].max().date()})")
print(f"test rows     : {n_test:,}  ({df_features.loc[df_features['is_test'], 'date'].min().date()} "
      f"to {df_features.loc[df_features['is_test'], 'date'].max().date()})")

assert n_train > 0 and n_test > 0, "empty split - check CUTOFF_DATE against the data range"

cutoff        : 2025-10-01
train rows    : 361,440  (2023-01-01 to 2025-09-30)
test rows     : 33,120  (2025-10-01 to 2025-12-31)


## Merge the external factors

Weather joins on `["date", "province"]`, holiday on `["date", "province"]`,
promotion on `promo_id`. Each join is asserted, because a key mismatch produces an
all-NaN column rather than an error.

In [7]:
df_features = df_features.merge(
    df_weather[["date", "province", "temperature", "humidity", "season",
                "temp_norm", "temp_anomaly", "is_humid"]],
    on=["date", "province"], how="left",
)
assert df_features["temperature"].isna().sum() == 0, "weather join failed"

holiday_cols = [c for c in
                ["date", "province", "is_public_holiday", "holiday_name",
                 "is_school_holiday", "days_to_holiday", "days_since_holiday",
                 "is_pre_holiday", "days_to_christmas"]
                if c in df_holiday.columns]
df_features = df_features.merge(df_holiday[holiday_cols],
                                on=["date", "province"], how="left")
assert df_features["is_public_holiday"].isna().sum() == 0, "holiday join failed"

df_features = df_features.merge(
    df_promotion[["promo_id", "promo_type", "duration_days"]],
    on="promo_id", how="left",
)
df_features["promo_type"] = df_features["promo_type"].fillna("None")
df_features["duration_days"] = df_features["duration_days"].fillna(0)

print(f"{len(df_features):,} rows x {df_features.shape[1]} cols after merges")
display(df_features.head(3))

394,560 rows x 28 cols after merges


,date,province,store_id,store_name,category,item_id,item_name,price,base_price,is_promotion,promo_id,sales,day_of_week,discount_pct,is_test,temperature,humidity,season,temp_norm,temp_anomaly,is_humid,is_public_holiday,holiday_name,is_school_holiday,days_to_holiday,days_since_holiday,promo_type,duration_days
0,2023-01-01,New South Wales,1,Sydney CBD,Staples,1,Long Grain Rice 2kg,5.5,5.5,False,0,10.0,6,0.0,False,18.3,70.2,Summer,22.804301,-4.5,0,True,New Year's Day,True,0,0,None,0.0
1,2023-01-02,New South Wales,1,Sydney CBD,Staples,1,Long Grain Rice 2kg,5.5,5.5,False,0,14.0,0,0.0,False,18.5,63.2,Summer,22.804301,-4.3,0,False,NaN,True,24,1,None,0.0
2,2023-01-03,New South Wales,1,Sydney CBD,Staples,1,Long Grain Rice 2kg,5.5,5.5,False,0,10.0,1,0.0,False,14.7,74.6,Summer,22.804301,-8.1,0,False,NaN,True,23,2,None,0.0


## Calendar features

Taken from `holiday_preprocessed.csv` rather than a hard-coded list. The holiday
calendar varies by state, and a national list would erase that variation.

If `is_pre_holiday` or `days_to_christmas` are absent from the preprocessed file,
they are derived here so the notebook runs either way. 

add `is_christmas_date` as sale volume is significant low on that date

In [8]:
df_features["year"] = df_features["date"].dt.year
df_features["month"] = df_features["date"].dt.month
df_features["day"] = df_features["date"].dt.day
df_features["day_of_week"] = df_features["date"].dt.dayofweek
df_features["is_weekend"] = (df_features["day_of_week"] >= 5).astype(int)
df_features["quarter"] = df_features["date"].dt.quarter
df_features["week_of_year"] = df_features["date"].dt.isocalendar().week.astype(int)
df_features["day_of_year"] = df_features["date"].dt.dayofyear


# Cyclical encodings, so December and January are adjacent rather than 11 apart.
df_features["month_sin"] = np.sin(2 * np.pi * df_features["month"] / 12)
df_features["month_cos"] = np.cos(2 * np.pi * df_features["month"] / 12)
df_features["dow_sin"] = np.sin(2 * np.pi * df_features["day_of_week"] / 7)
df_features["dow_cos"] = np.cos(2 * np.pi * df_features["day_of_week"] / 7)

if "days_to_christmas" not in df_features.columns:
    xmas = pd.to_datetime(df_features["year"].astype(str) + "-12-25")
    d2x = (xmas - df_features["date"]).dt.days
    df_features["days_to_christmas"] = d2x.where((d2x >= 0) & (d2x <= 30), 99).astype(int)

df_features["is_christmas_date"] = (df_features["days_to_christmas"] == 0).astype(int)

if "is_pre_holiday" not in df_features.columns:
    df_features["is_pre_holiday"] = (df_features["days_to_holiday"] == 1).astype(int)

print([c for c in df_features.columns if "holiday" in c or "christmas" in c])

['is_public_holiday', 'holiday_name', 'is_school_holiday', 'days_to_holiday', 'days_since_holiday', 'days_to_christmas', 'is_christmas_date', 'is_pre_holiday']


## Weather features

The original bins `[-inf, 20, 25, 30, inf]` were tuned for a tropical climate. On
Australian data they put 62% of rows in "Cold" and 0.5% in "Hot", which is close to
useless as a split.

Two changes. Bins are cut on the **temperature anomaly** rather than the raw value,
so a hot day means hot *for that city in that month*. And the edges are quantiles
**computed on the training window only**.

In [9]:
train_mask = ~df_features["is_test"]

# Quantile edges from training data only.
anom_q = df_features.loc[train_mask, "temp_anomaly"].quantile([0.25, 0.5, 0.75]).values
hum_q = df_features.loc[train_mask, "humidity"].quantile([0.33, 0.67]).values
print(f"temp_anomaly edges (train only): {np.round(anom_q, 2)}")
print(f"humidity edges (train only)    : {np.round(hum_q, 2)}")

df_features["temp_category"] = pd.cut(
    df_features["temp_anomaly"],
    bins=[-np.inf, *anom_q, np.inf],
    labels=["Much cooler", "Cooler", "Warmer", "Much warmer"],
)
df_features["humidity_level"] = pd.cut(
    df_features["humidity"],
    bins=[-np.inf, *hum_q, np.inf],
    labels=["Low", "Medium", "High"],
)

display(df_features["temp_category"].value_counts(normalize=True).round(3))

temp_anomaly edges (train only): [-2.3  -0.03  2.31]
humidity edges (train only)    : [61.5 69.6]


temp_category
Much warmer    0.252
Warmer         0.250
Cooler         0.250
Much cooler    0.248
Name: proportion, dtype: float64

In [10]:
# Heat and cold excess against the local norm - captures the non-linear response
# that a linear temperature term misses.
df_features["heat_excess"] = np.maximum(0, df_features["temperature"] - 28)
df_features["cold_excess"] = np.maximum(0, 14 - df_features["temperature"])

display(df_features[["temperature", "temp_anomaly", "heat_excess",
                     "cold_excess"]].describe().round(2))

,temperature,temp_anomaly,heat_excess,cold_excess
count,394560.00,394560.00,394560.00,394560.00
mean,18.12,0.00,0.03,0.81
std,5.57,3.55,0.27,2.04
min,-1.00,-15.20,0.00,0.00
25%,14.40,-2.27,0.00,0.00
50%,18.30,-0.00,0.00,0.00
75%,22.30,2.33,0.00,0.00
max,34.10,13.88,6.10,15.00


## Price and promotion features

The original notebook ignored these columns entirely, yet promotion is the largest
single driver in this dataset.

In [11]:
df_features["log_price_ratio"] = np.log(
    np.maximum(df_features["price"] / df_features["base_price"], 1e-6)
)
df_features["is_deep_discount"] = (df_features["discount_pct"] >= 30).astype(int)

# Rival price index: mean discount on other items in the same category, same store,
# same day. A crude cannibalisation proxy.
grp = df_features.groupby(["store_id", "category", "date"])["discount_pct"]
n_in_group = grp.transform("count")
df_features["rival_discount_pct"] = np.where(
    n_in_group > 1,
    (grp.transform("sum") - df_features["discount_pct"]) / np.maximum(n_in_group - 1, 1),
    0.0,
)

display(df_features[["price", "base_price", "discount_pct", "log_price_ratio",
                     "rival_discount_pct"]].describe().round(3))

,price,base_price,discount_pct,log_price_ratio,rival_discount_pct
count,394560.000,394560.000,394560.000,394560.000,394560.000
mean,6.227,6.453,3.352,-0.039,3.352
std,3.966,4.079,8.262,0.099,3.796
min,1.140,2.200,0.000,-0.697,0.000
25%,3.980,4.250,0.000,0.000,0.000
50%,5.050,5.220,0.000,0.000,2.500
75%,7.350,7.580,0.000,0.000,5.020
max,25.100,25.100,50.200,0.000,33.750


## Lag features

In [12]:
df_features = df_features.sort_values(["store_id", "item_id", "date"]).reset_index(drop=True)
df_features["store_item"] = (
    df_features["store_id"].astype(str) + "_" + df_features["item_id"].astype(str)
)

for n in [1, 7, 14, 28]:
    df_features[f"sales_lag_{n}"] = (
        df_features.groupby("store_item")["sales"].shift(n)
    )

print([c for c in df_features.columns if c.startswith("sales_lag")])

['sales_lag_1', 'sales_lag_7', 'sales_lag_14', 'sales_lag_28']


### Rolling window features

In [13]:
g = df_features.groupby("store_item")["sales"]
shifted = g.shift(1)

for window in [7, 14, 28]:
    roll = shifted.groupby(df_features["store_item"]).rolling(window, min_periods=1)
    df_features[f"sales_mean_{window}d"] = roll.mean().reset_index(level=0, drop=True)
    df_features[f"sales_min_{window}d"] = roll.min().reset_index(level=0, drop=True)
    df_features[f"sales_max_{window}d"] = roll.max().reset_index(level=0, drop=True)
    df_features[f"sales_std_{window}d"] = roll.std().reset_index(level=0, drop=True)

print([c for c in df_features.columns if "sales_mean" in c or "sales_std" in c])

['sales_mean_7d', 'sales_std_7d', 'sales_mean_14d', 'sales_std_14d', 'sales_mean_28d', 'sales_std_28d']


### Exponentially weighted moving average

In [14]:
for alpha in [0.5, 0.75]:
    a_str = str(alpha).replace(".", "")
    df_features[f"sales_ewma_a{a_str}"] = (
        shifted.groupby(df_features["store_item"])
        .apply(lambda x: x.ewm(alpha=alpha).mean())
        .reset_index(level=0, drop=True)
    )

print([c for c in df_features.columns if "ewma" in c])

['sales_ewma_a05', 'sales_ewma_a075']


In [15]:
# Spot-check one real series from this dataset.
example = df_features["store_item"].iloc[0]
display(
    df_features[df_features["store_item"] == example]
    [["date", "store_name", "item_name", "sales", "sales_lag_1",
      "sales_mean_7d", "sales_ewma_a05"]]
    .head(10)
)

,date,store_name,item_name,sales,sales_lag_1,sales_mean_7d,sales_ewma_a05
0,2023-01-01,Sydney CBD,Long Grain Rice 2kg,10.0,NaN,NaN,NaN
1,2023-01-02,Sydney CBD,Long Grain Rice 2kg,14.0,10.0,10.000000,10.000000
2,2023-01-03,Sydney CBD,Long Grain Rice 2kg,10.0,14.0,12.000000,12.666667
3,2023-01-04,Sydney CBD,Long Grain Rice 2kg,14.0,10.0,11.333333,11.142857
4,2023-01-05,Sydney CBD,Long Grain Rice 2kg,17.0,14.0,12.000000,12.666667
5,2023-01-06,Sydney CBD,Long Grain Rice 2kg,19.0,17.0,13.000000,14.903226
6,2023-01-07,Sydney CBD,Long Grain Rice 2kg,16.0,19.0,14.000000,16.984127
7,2023-01-08,Sydney CBD,Long Grain Rice 2kg,17.0,16.0,14.285714,16.488189
8,2023-01-09,Sydney CBD,Long Grain Rice 2kg,18.0,17.0,15.285714,16.745098
9,2023-01-10,Sydney CBD,Long Grain Rice 2kg,14.0,18.0,15.857143,17.373777


## Store-level and item-level features

The original version was:

```python
df_features["store_mean_7d"] = df_features.groupby(["store_name", "date"])["sales"].transform("mean")
```

That is the **same-day** cross-sectional mean across all items in the store, and it
includes the current row's own sales. It is not a 7-day lag despite the name, and it
leaks the target: correlation with `sales` measured at 0.25 on this data.

The corrected version shifts by one day first, so only past information is used.

In [16]:
# Daily aggregate per store, then lagged and rolled.
store_daily = (
    df_features.groupby(["store_id", "date"])["sales"].mean()
    .rename("store_daily_mean").reset_index().sort_values(["store_id", "date"])
)
store_daily["store_mean_7d"] = (
    store_daily.groupby("store_id")["store_daily_mean"]
    .transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
)

item_daily = (
    df_features.groupby(["item_id", "date"])["sales"].mean()
    .rename("item_daily_mean").reset_index().sort_values(["item_id", "date"])
)
item_daily["item_mean_7d"] = (
    item_daily.groupby("item_id")["item_daily_mean"]
    .transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
)

df_features = df_features.merge(
    store_daily[["store_id", "date", "store_mean_7d"]], on=["store_id", "date"], how="left"
)
df_features = df_features.merge(
    item_daily[["item_id", "date", "item_mean_7d"]], on=["item_id", "date"], how="left"
)

print("corr(store_mean_7d, sales):", round(df_features["store_mean_7d"].corr(df_features["sales"]), 4))
print("corr(item_mean_7d,  sales):", round(df_features["item_mean_7d"].corr(df_features["sales"]), 4))
print("(both should be well below the 0.25 of the same-day version)")

corr(store_mean_7d, sales): 0.1968
corr(item_mean_7d,  sales): 0.8511
(both should be well below the 0.25 of the same-day version)


## Categorical encoding

`get_dummies` on the full frame derives its category list from train and test
together. Here the columns are left as `category` dtype instead, so the encoder can
be fitted inside the modelling pipeline on the training window.

LightGBM and XGBoost both accept `category` dtype natively, so for the tree models
no encoding is needed at all.

In [17]:
cat_cols = ["province", "store_name", "category", "item_name", "season",
            "promo_type", "temp_category", "humidity_level", "holiday_name"]
for c in cat_cols:
    if c in df_features.columns:
        df_features[c] = df_features[c].astype("category")

print(df_features[cat_cols].dtypes.to_string())
print("\nLightGBM and XGBoost consume these directly via enable_categorical=True.")

province          category
store_name        category
category          category
item_name         category
season            category
promo_type        category
temp_category     category
humidity_level    category
holiday_name      category

LightGBM and XGBoost consume these directly via enable_categorical=True.


## Drop warm-up rows

`dropna()` on the whole frame removes any row with a missing value anywhere. Only
the lag warm-up period needs dropping, so restrict it to the lag columns.

In [18]:
lag_cols = [c for c in df_features.columns
            if c.startswith(("sales_lag_", "sales_mean_", "sales_min_",
                             "sales_max_", "sales_std_", "sales_ewma_"))]

before = len(df_features)
df_features = df_features.dropna(subset=lag_cols)
print(f"dropped {before - len(df_features):,} warm-up rows, {len(df_features):,} remain")
print(f"first usable date: {df_features['date'].min().date()}")
print(f"\ntest rows surviving: {int(df_features['is_test'].sum()):,}")
assert df_features["is_test"].sum() > 0, "test set emptied - check the drop"

dropped 10,080 warm-up rows, 384,480 remain
first usable date: 2023-01-29

test rows surviving: 33,120


## Drop is_promotion column
drop is_promotion column as it is highly correlative to discount_pct column

In [19]:
df_features = df_features.drop(columns='is_promotion')

## Save feature engineered data

In [20]:
df_features.info()

<class 'pandas.core.frame.DataFrame'>
Index: 384480 entries, 28 to 394559
Data columns (total 69 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   date                384480 non-null  datetime64[ns]
 1   province            384480 non-null  category      
 2   store_id            384480 non-null  int64         
 3   store_name          384480 non-null  category      
 4   category            384480 non-null  category      
 5   item_id             384480 non-null  int64         
 6   item_name           384480 non-null  category      
 7   price               384480 non-null  float64       
 8   base_price          384480 non-null  float64       
 9   promo_id            384480 non-null  int64         
 10  sales               384480 non-null  float64       
 11  day_of_week         384480 non-null  int32         
 12  discount_pct        384480 non-null  float64       
 13  is_test             384480 non-nu

In [21]:
num_features = df_features.shape[1]
save_path = os.path.join(DATA_DIR, f"feature_engineered_data_{num_features}_features.parquet")

df_features.to_parquet(save_path, index=False)
print(f"saved {len(df_features):,} rows x {num_features} cols")
print(save_path)

saved 384,480 rows x 69 cols
../data\feature_engineered_data_69_features.parquet


## What was changed, and why

| Issue in the original | Fix |
|---|---|
| `cutoff_date = 2017-10-01` against 2023-2025 data - the entire set became test and train was empty | `CUTOFF_DATE = 2025-10-01`, asserted non-empty on both sides |
| Hard-coded Vietnamese holiday list | Joined from `holiday_preprocessed.csv`, preserving state-level variation |
| Temperature bins `[20, 25, 30]` tuned for a tropical climate - 62% "Cold", 0.5% "Hot" | Quantile bins on `temp_anomaly`, edges fitted on train only |
| `get_dummies` on the full frame | `category` dtype, encoding deferred to the modelling pipeline |
| `store_mean_7d` used the same-day cross-sectional mean including the row's own sales | Shifted one day, then rolled over 7 days |
| `price`, `is_promotion`, `discount_pct`, `promo_type` unused | Price ratio, deep-discount flag, rival discount index, promotion type |
| Spot-check on `'Ba Dinh Supermarket_Baby Wipes'` | A series that exists in this dataset |
| `dropna()` across all columns | Restricted to lag warm-up columns |
| Saved as `.feather` via a helper | `.parquet` directly, no external dependency |